# Scientific ShapeNetCore evaluation
Run all mặc định đánh giá hai view synthetic theo thứ tự và chạy lại để xác minh resume. Notebook kiểm tra GPU, mount Drive trên Colab và ghi từng kết quả tăng dần. Production dùng `chair_colab.yaml` cùng cache backend thật.

In [ ]:
import os, shutil, subprocess, sys, time
from pathlib import Path
STARTED = time.perf_counter()
cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [cwd, cwd / 'image-to-3d-shapenet', *cwd.parents] if (p / 'configs' / 'quick_test.yaml').is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Run from the cloned repository or its parent.')
try:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_ROOT / 'requirements-colab.txt')], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT_ROOT)], check=True)
except subprocess.CalledProcessError as exc:
    raise RuntimeError('Cài dependency thất bại; Restart runtime rồi Run all lại.') from exc
import torch
CUDA = torch.cuda.is_available()
DRIVE_ROOT = None
try:
    from google.colab import drive
    mount = os.getenv('GOOGLE_DRIVE_MOUNT', '/content/drive')
    drive.mount(mount)
    DRIVE_ROOT = Path(mount) / 'MyDrive' / 'image-to-3d-shapenet'
except ImportError:
    pass
HF_TOKEN_AVAILABLE = bool(os.getenv('HF_TOKEN'))
try:
    from google.colab import userdata
    HF_TOKEN_AVAILABLE = HF_TOKEN_AVAILABLE or bool(userdata.get('HF_TOKEN'))
except Exception:
    pass
print({'cuda': CUDA, 'gpu': torch.cuda.get_device_name(0) if CUDA else None, 'vram_gb': round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2) if CUDA else 0.0, 'hf_token_available': HF_TOKEN_AVAILABLE, 'drive': str(DRIVE_ROOT) if DRIVE_ROOT else None})
CONFIG = PROJECT_ROOT / 'configs' / 'quick_test.yaml'
CHECKPOINT_DIR = (DRIVE_ROOT / 'evaluation_checkpoint') if DRIVE_ROOT else (PROJECT_ROOT / 'outputs' / 'quick_test' / 'evaluation_checkpoint')
EVALUATION_DIR = (DRIVE_ROOT / 'notebook_evaluation') if DRIVE_ROOT else (PROJECT_ROOT / 'outputs' / 'quick_test' / 'notebook_evaluation')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
EVALUATION_DIR.mkdir(parents=True, exist_ok=True)
def run_checked(command, label):
    try:
        subprocess.run(command, cwd=PROJECT_ROOT, check=True)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(f'{label} thất bại; xem log ngay phía trên và kiểm tra RAM/disk/checkpoint.') from exc

In [ ]:
for script in ('prepare_dataset.py', 'render_dataset.py'):
    run_checked([sys.executable, str(PROJECT_ROOT / 'scripts' / script), '--config', str(CONFIG)], script)
run_checked([sys.executable, str(PROJECT_ROOT / 'scripts' / 'generate_base_mesh_cache.py'), '--config', str(CONFIG), '--max-samples', '4'], 'base mesh cache')
checkpoint = CHECKPOINT_DIR / 'best_edge.pt'
if not checkpoint.is_file():
    run_checked([sys.executable, str(PROJECT_ROOT / 'scripts' / 'train.py'), '--config', str(CONFIG), '--quick-overfit', '--checkpoint-dir', str(CHECKPOINT_DIR)], 'quick train')
assert checkpoint.is_file()

In [ ]:
command = [sys.executable, str(PROJECT_ROOT / 'scripts' / 'evaluate.py'), '--config', str(CONFIG), '--checkpoint', str(checkpoint), '--output', str(EVALUATION_DIR)]
run_checked(command, 'evaluation')
# Running again verifies incremental resume: completed rows are skipped.
run_checked(command, 'evaluation resume')

In [ ]:
import csv, json
from PIL import Image
summary = list(csv.DictReader((EVALUATION_DIR / 'metrics_summary.csv').open(encoding='utf-8')))
display(summary)
print((EVALUATION_DIR / 'benchmark_table.md').read_text(encoding='utf-8'))
print('failures:', json.loads((EVALUATION_DIR / 'failures.json').read_text(encoding='utf-8')))
display(Image.open(EVALUATION_DIR / 'plots' / 'chamfer_distance.png'))
comparison = sorted((EVALUATION_DIR / 'comparisons').glob('*refiner_full.png'))[0]
display(Image.open(comparison))
disk = shutil.disk_usage(PROJECT_ROOT)
peak_vram = torch.cuda.max_memory_allocated() / 1024**3 if CUDA else 0.0
print({'elapsed_seconds': round(time.perf_counter() - STARTED, 2), 'peak_vram_gb': round(peak_vram, 3), 'disk_free_gb': round(disk.free / 1024**3, 2), 'resume_rows': len(list(csv.DictReader((EVALUATION_DIR / 'metrics_per_sample.csv').open(encoding='utf-8'))))})